In [0]:
# Tags on raw tables do not carry over to new tables, so the published columns are tagged again here.
# Ungoverned tags (discovery, lineage, audit): they do not count against the account's governed-tag quota.
# They are not used by ABAC policies.
dbutils.widgets.text("catalog", "serverless_stable_yit814_catalog")
catalog = dbutils.widgets.get("catalog")
ANALYTICS, GOV = f"{catalog}.pii_analytics", f"{catalog}.pii_governance"

protection = {"encrypt": [("_enc", "encrypted_aes_gcm"), ("_hmac", "hmac_sha256")], "tokenize": [("_token", "tokenized_vault")]}
results = []
for r in spark.sql(f"SELECT table_name, column_name, pii_type, method FROM {GOV}.pii_contract WHERE active AND method IN ('encrypt', 'tokenize')").collect():
    for suffix, value in protection[r.method]:
        table, column = f"{ANALYTICS}.{r.table_name}", f"{r.column_name}{suffix}"
        tags = f"'pii_protection' = '{value}', 'source_pii_type' = '{r.pii_type}'"
        status = "tagged"
        try:
            spark.sql(f"ALTER MATERIALIZED VIEW {table} ALTER COLUMN `{column}` SET TAGS ({tags})")
        except Exception as e:
            status = f"FAILED: {str(e)[:200]}"
        results.append((table, column, value, status))

for t in ["customers", "transactions", "customer_spend_by_segment"]:
    try:
        spark.sql(f"ALTER MATERIALIZED VIEW {ANALYTICS}.{t} SET TAGS ('data_zone' = 'pseudonymized')")
    except Exception as e:
        results.append((f"{ANALYTICS}.{t}", "<table>", "data_zone", f"FAILED: {str(e)[:200]}"))
display(spark.createDataFrame(results, "table STRING, column STRING, tag STRING, status STRING"))
assert all(x[3] == "tagged" for x in results), "Some output tags could not be applied"